# Exploring Jumps in CDS, Oil, and MSCI Time Series

**Methodology:** Characterizing Jumps via GARCH(1,1) vs. GARCH(1,1)-Jump

---

**Steps:**

1. Prepare the return series (log returns for CDS, Oil, and MSCI)
2. Estimate plain GARCH(1,1) and inspect residuals
3. Estimate GARCH(1,1)-Jump and inspect residuals
4. Compare both models (likelihood ratio test, information criteria, parameter shifts, residual improvement)
5. Extract jump parameters ($\lambda$, $\theta$, $\delta^2$)
6. Cross-country comparison (oil exporters vs. controls)

## Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import poisson, norm, chi2, fisher_exact, ttest_ind, mannwhitneyu
from scipy.special import gammaln
import matplotlib.pyplot as plt
import datetime
import warnings
warnings.filterwarnings('ignore')
from numba import njit

## Load Data

In [ ]:
# Daily CDS (already weekly)
CDS_data = pd.read_csv('..data/processed/CDS/Daily_CDS.csv')
CDS_data['Date'] = pd.to_datetime(CDS_data['Date'])
CDS_data.set_index('Date', inplace=True)

# Oil prices (daily → weekly)
Oil_data = pd.read_csv('data/processed/Oil/oil_prices_datastream.csv')
Oil_data['Date'] = pd.to_datetime(Oil_data['Date'])
Oil_data.set_index('Date', inplace=True)
#Oil_data = Oil_data.resample('W-FRI').last()

# MSCI country indices (daily → weekly)
MSCI_data = pd.read_csv('data/processed/MSCI_indices/mscicountryindex.csv')  # adjust path as needed
MSCI_data['Date'] = pd.to_datetime(MSCI_data['Date'])
MSCI_data.set_index('Date', inplace=True)
#MSCI_data = MSCI_data.resample('W-FRI').last()

# Filter to 2014+
CDS_data = CDS_data[CDS_data.index >= datetime.datetime(2014, 1, 1)]
Oil_data = Oil_data[Oil_data.index >= datetime.datetime(2014, 1, 1)]
MSCI_data = MSCI_data[MSCI_data.index >= datetime.datetime(2014, 1, 1)]

### Country Lists

In [46]:
# MSCI EM constituents (display names)
MSCI_EM_constituent_list = [
    "Brazil", "Chile", "China", "Colombia", "Czechia", "Egypt",
    "Greece", "Hungary", "India", "Indonesia", "South Korea", "Kuwait",
    "Malaysia", "Mexico", "Peru", "Philippines", "Poland", "Qatar",
    "Saudi Arabia", "South Africa", "Taiwan", "Thailand", "Turkey", "United Arab Emirates"
]

# Cleaned names matching our data columns (UAE -> Abu Dhabi + Dubai for CDS)
MSCI_EM_constituent_list_clean = [
    "Brazil", "Chile", "China", "Colombia", "Czechia", "Egypt",
    "Greece", "Hungary", "India", "Indonesia", "South Korea", "Kuwait",
    "Malaysia", "Mexico", "Peru", "Philippines", "Poland", "Qatar",
    "Saudi Arabia", "South Africa", "Taiwan", "Thailand", "Turkey", "Abu Dhabi", "Dubai"
]

# Note: UAE has MSCI data but no single CDS — Abu Dhabi and Dubai have separate CDS

oil_exporters = [
    'Saudi Arabia', 'Qatar', 'Abu Dhabi', 'Dubai', 'Kuwait',
    'Colombia', 'Mexico', 'Brazil', 'Malaysia', 'Indonesia',
    'Kazakhstan', 'Norway', 'Egypt'
]

---

## 1. Prepare the Return Series

We compute log returns from each raw price or spread series:

$$r_t = \ln(P_t) - \ln(P_{t-1})$$

For daily data, each $r_t$ represents one day's log change. This transformation is applied to:
- The oil price series (Brent, WTI, OPEC basket, Dubai Crude)
- Each sovereign CDS spread series
- Each MSCI country index series

In [47]:
# Oil returns
oil_returns = np.log(Oil_data / Oil_data.shift(1)).dropna()

# CDS returns
cds_returns = np.log(CDS_data / CDS_data.shift(1)).dropna()

# MSCI returns
msci_returns = np.log(MSCI_data / MSCI_data.shift(1)).dropna()

print(f"Oil series: {oil_returns.shape[0]} obs, benchmarks: {list(oil_returns.columns)}")
print(f"CDS series: {cds_returns.shape[0]} obs, countries: {cds_returns.shape[1]}")
print(f"MSCI series: {msci_returns.shape[0]} obs, countries: {msci_returns.shape[1]}")

Oil series: 2868 obs, benchmarks: ['Brent', 'WTI', 'OPEC_basket', 'Dubai_Crude']
CDS series: 1970 obs, countries: 72
MSCI series: 2869 obs, countries: 85


### Quick Visual: CDS Returns

In [ ]:
def plot_returns(names, returns_df, label='Returns', figsize=(12, 4)):
    """
    Plot returns for one or more series.

    Parameters
    ----------
    names : str or list
        Single name or list of column names.
    returns_df : DataFrame
        Returns with series as columns.
    label : str
        Y-axis label.
    figsize : tuple
        Figure size per subplot.
    """
    if isinstance(names, str):
        names = [names]

    n = len(names)
    fig, axes = plt.subplots(n, 1, figsize=(figsize[0], figsize[1] * n), squeeze=False)

    for i, name in enumerate(names):
        ax = axes[i, 0]
        ret = returns_df[name].dropna()

        ax.plot(ret.index, ret.values, linewidth=0.5, color='steelblue')
        ax.axhline(0, color='black', linewidth=0.5)

        zero_pct = (ret == 0).sum() / len(ret) * 100
        ax.set_title(f"{name}  |  std={ret.std()*100:.2f}%  |  zeros={zero_pct:.0f}%")
        ax.set_ylabel(label)

    axes[-1, 0].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

plot_returns(['Brazil', 'Saudi Arabia', 'Dubai', 'Turkey'], cds_returns, label='CDS Returns')

---

## 2. Estimate the Plain GARCH(1,1)

We fit the standard GARCH(1,1) model, which captures volatility clustering through a smooth, persistent variance process.

**Mean equation:**

$$r_t = \mu + \varepsilon_t, \quad \varepsilon_t = \sqrt{h_t} \cdot z_t, \quad z_t \sim N(0,1)$$

**Variance equation:**

$$h_t = \omega + \alpha \, \varepsilon_{t-1}^2 + \beta \, h_{t-1}$$

The parameter set is $\{\mu, \omega, \alpha, \beta\}$, estimated via maximum likelihood. We store:

- The maximized log-likelihood $\mathcal{L}_{\text{GARCH}}$
- The parameter estimates $\{\hat{\mu}, \hat{\omega}, \hat{\alpha}, \hat{\beta}\}$
- The standardized residuals $\hat{z}_t = \varepsilon_t / \sqrt{h_t}$

**Diagnostic checks on residuals:**

- Kurtosis (expect > 3 if jumps are present)
- Jarque-Bera test for normality
- QQ-plot against the normal distribution

If the series contains genuine jumps, the plain GARCH must absorb extreme observations by inflating $\alpha$, and the residuals will exhibit fat tails.

In [61]:
@njit
def garch_loglik_numba(params, returns):
    """GARCH(1,1) log-likelihood — numba accelerated."""
    mu, omega, alpha, beta = params
    T = len(returns)

    h = np.zeros(T)
    h[0] = np.var(returns)

    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]

    ll = 0.0
    for t in range(T):
        ll += -0.5 * (np.log(2 * np.pi * h[t]) + (returns[t] - mu)**2 / h[t])

    return -ll  # negative for minimization


def estimate_garch(returns):
    """Estimate GARCH(1,1) via MLE."""
    x0 = [np.mean(returns), 1e-5, 0.05, 0.90]

    bounds = [
        (None, None),      # mu
        (1e-8, None),      # omega > 0
        (1e-6, 1.0),       # alpha
        (1e-4, 0.999)       # beta
    ]

    result = minimize(garch_loglik_numba, x0, args=(returns,), method='L-BFGS-B',
                      bounds=bounds, options={'maxiter': 2000})

    return {
        'params': result.x,
        'loglik': -result.fun,
        'converged': result.success
    }

---

## 3. Estimate the GARCH(1,1)-Jump Model

We augment the model by decomposing returns into a smooth (continuous) component and a discrete jump component.

**Mean equation:**

$$r_t = \mu + \sqrt{h_t} \cdot z_t + \sum_{k=1}^{n_t} J_{t,k}$$

where:

- $n_t \sim \text{Poisson}(\lambda)$ is the number of jumps at time $t$
- Each jump $J_{t,k} \sim N(\theta, \delta^2)$ with mean $\theta$ and variance $\delta^2$
- $z_t \sim N(0,1)$ is the continuous innovation

**Variance equation (same structure):**

$$h_t = \omega + \alpha \, \varepsilon_{t-1}^2 + \beta \, h_{t-1}$$

where $\varepsilon_{t-1}$ now refers only to the continuous innovation, not the total return surprise. The parameter set expands to $\{\mu, \omega, \alpha, \beta, \lambda, \theta, \delta^2\}$ — three additional parameters.

**Likelihood function:**

Since we do not observe whether a jump occurred at time $t$, we sum over the Poisson-weighted possibilities. The density of $r_t$ conditional on the information set $\mathcal{F}_{t-1}$ is:

$$f(r_t \mid \mathcal{F}_{t-1}) = \sum_{j=0}^{J_{\max}} \frac{e^{-\lambda} \lambda^j}{j!} \cdot \frac{1}{\sqrt{h_t + j\delta^2}} \cdot \phi\!\left(\frac{r_t - \mu - j\theta}{\sqrt{h_t + j\delta^2}}\right)$$

where $\phi(\cdot)$ is the standard normal density. We truncate at $J_{\max} = 10$ since the Poisson probability of more jumps per period is negligible at daily frequency.

The total log-likelihood is:

$$\mathcal{L}_{\text{GARCH-J}} = \sum_{t=1}^{T} \ln \, f(r_t \mid \mathcal{F}_{t-1})$$

maximized over all seven parameters using numerical optimization.

In [62]:
@njit
def garch_jump_loglik_numba(params, returns, max_jumps=10):
    """GARCH(1,1)-Jump log-likelihood with Poisson jumps — numba accelerated."""
    mu, omega, alpha, beta, lam, theta, delta = params
    T = len(returns)

    # GARCH recursion
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]

    # Likelihood
    ll = 0.0
    for t in range(T):
        prob_t = 0.0
        log_poisson = -lam
        for k in range(max_jumps + 1):
            var_k = h[t] + k * delta**2
            mean_k = mu + k * theta
            pdf_k = np.exp(-0.5 * (returns[t] - mean_k)**2 / var_k) / np.sqrt(2 * np.pi * var_k)
            poisson_k = np.exp(log_poisson)
            prob_t += pdf_k * poisson_k
            log_poisson += np.log(lam) - np.log(k + 1)  # update for next k
        ll += np.log(prob_t + 1e-10)

    return -ll  # negative for minimization


def estimate_garch_jump(returns):
    """Estimate GARCH(1,1)-Jump via MLE."""
    x0 = [np.mean(returns), 1e-5, 0.05, 0.85, 0.03, -0.01, 0.03]

    bounds = [
        (None, None),      # mu
        (1e-8, None),      # omega > 0
        (1e-6, 1.0),       # alpha
        (1e-4, 0.999),      # beta
        (1e-4, 1.5),       # lambda (jump intensity)
        (-0.2, 0.2),       # theta (jump mean)
        (1e-4, 0.5)        # delta (jump std)
    ]

    result = minimize(garch_jump_loglik_numba, x0, args=(returns,), method='L-BFGS-B',
                      bounds=bounds, options={'maxiter': 2000})

    return {
        'params': result.x,
        'loglik': -result.fun,
        'converged': result.success
    }

---

## 4. Compare the Two Models

Since the plain GARCH is nested within the GARCH-Jump model (set $\lambda = 0$), we use the following comparisons:

**4a. Likelihood Ratio Test**

$$LR = 2\left(\mathcal{L}_{\text{GARCH-J}} - \mathcal{L}_{\text{GARCH}}\right) \sim \chi^2(3)$$

The three degrees of freedom correspond to the three extra parameters $(\lambda, \theta, \delta^2)$. We reject the null of no jumps if $LR > 7.81$ at the 5% level.

*Note:* Testing $\lambda = 0$ lies on the boundary of the parameter space, which can affect the asymptotic distribution. The standard $\chi^2$ test serves as a reasonable approximation.

**4b. Information Criteria**

$$\text{AIC} = -2\mathcal{L} + 2k, \quad \text{BIC} = -2\mathcal{L} + k \ln T$$

If the GARCH-Jump model achieves lower AIC/BIC despite the penalty for extra parameters, the evidence for jumps is robust.

**4c. Parameter Shifts**

We compare $\hat{\alpha}$ across models. If jumps are present: $\hat{\alpha}_{\text{GARCH-J}} < \hat{\alpha}_{\text{GARCH}}$

**4d. Residual Improvement**

Reduced kurtosis, improved QQ-plot, Jarque-Bera statistic closer to zero.

### 4.1 Oil Benchmarks

In [64]:
for benchmark in ['Brent', 'WTI', 'OPEC_basket', 'Dubai_Crude']:

    oil_ret = Oil_data[benchmark].pct_change().dropna().values

    garch_oil = estimate_garch(oil_ret)
    garch_jump_oil = estimate_garch_jump(oil_ret)

    T = len(oil_ret)
    k_garch, k_jump = 4, 7

    # LR test
    LR_oil = 2 * (garch_jump_oil['loglik'] - garch_oil['loglik'])
    p_oil = 1 - chi2.cdf(LR_oil, df=3)

    # Information criteria
    aic_garch = -2 * garch_oil['loglik'] + 2 * k_garch
    bic_garch = -2 * garch_oil['loglik'] + k_garch * np.log(T)
    aic_jump  = -2 * garch_jump_oil['loglik'] + 2 * k_jump
    bic_jump  = -2 * garch_jump_oil['loglik'] + k_jump * np.log(T)

    print("=" * 60)
    print(f"{benchmark} OIL")
    print("=" * 60)
    print(f"GARCH LogLik:      {garch_oil['loglik']:.2f}   (AIC={aic_garch:.1f}, BIC={bic_garch:.1f})")
    print(f"GARCH-Jump LogLik: {garch_jump_oil['loglik']:.2f}   (AIC={aic_jump:.1f}, BIC={bic_jump:.1f})")
    print(f"LR statistic:      {LR_oil:.2f}")
    print(f"p-value:           {p_oil:.6f}")
    print(f"\nGARCH alpha:       {garch_oil['params'][2]:.4f}")
    print(f"Jump  alpha:       {garch_jump_oil['params'][2]:.4f}  (drop = {garch_oil['params'][2] - garch_jump_oil['params'][2]:.4f})")
    print(f"\nJump parameters:")
    print(f"  lambda = {garch_jump_oil['params'][4]:.4f} ({garch_jump_oil['params'][4]*252:.1f} jumps/year)")
    print(f"  theta  = {garch_jump_oil['params'][5]:.4f}")
    print(f"  delta  = {garch_jump_oil['params'][6]:.4f}")
    print()

Brent OIL
GARCH LogLik:      7156.14   (AIC=-14304.3, BIC=-14280.4)
GARCH-Jump LogLik: 7288.17   (AIC=-14562.3, BIC=-14520.6)
LR statistic:      264.07
p-value:           0.000000

GARCH alpha:       0.0500
Jump  alpha:       0.0946  (drop = -0.0446)

Jump parameters:
  lambda = 0.0397 (10.0 jumps/year)
  theta  = -0.0119
  delta  = 0.0446

WTI OIL
GARCH LogLik:      6336.14   (AIC=-12664.3, BIC=-12640.4)
GARCH-Jump LogLik: 6865.03   (AIC=-13716.1, BIC=-13674.3)
LR statistic:      1057.78
p-value:           0.000000

GARCH alpha:       0.0972
Jump  alpha:       0.1148  (drop = -0.0176)

Jump parameters:
  lambda = 0.0594 (15.0 jumps/year)
  theta  = -0.0212
  delta  = 0.0378

OPEC_basket OIL
GARCH LogLik:      7582.80   (AIC=-15157.6, BIC=-15133.8)
GARCH-Jump LogLik: 7656.18   (AIC=-15298.4, BIC=-15256.6)
LR statistic:      146.74
p-value:           0.000000

GARCH alpha:       0.0881
Jump  alpha:       0.1044  (drop = -0.0163)

Jump parameters:
  lambda = 0.0269 (6.8 jumps/year)
  the

### 4.2 CDS Spread Series

In [65]:
cds_results = []

for country in MSCI_EM_constituent_list_clean:
    if country not in cds_returns.columns:
        print(f"{country}: NOT FOUND in CDS data — skipped")
        continue

    ret = cds_returns[country].dropna().values

    if len(ret) < 200:
        print(f"{country}: SKIPPED (only {len(ret)} obs)")
        continue

    zero_pct = (ret == 0).sum() / len(ret)
    if zero_pct > 0.3:
        print(f"{country}: SKIPPED ({zero_pct*100:.0f}% zeros)")
        continue

    if np.var(ret) < 1e-10:
        print(f"{country}: SKIPPED (no variance)")
        continue

    garch = estimate_garch(ret)
    garch_jump = estimate_garch_jump(ret)

    T = len(ret)
    k_garch, k_jump = 4, 7

    LR = 2 * (garch_jump['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    aic_garch = -2 * garch['loglik'] + 2 * k_garch
    bic_garch = -2 * garch['loglik'] + k_garch * np.log(T)
    aic_jump  = -2 * garch_jump['loglik'] + 2 * k_jump
    bic_jump  = -2 * garch_jump['loglik'] + k_jump * np.log(T)

    cds_results.append({
        'Country': country,
        'T': T,
        'GARCH_LL': garch['loglik'],
        'Jump_LL': garch_jump['loglik'],
        'AIC_GARCH': aic_garch,
        'AIC_Jump': aic_jump,
        'BIC_GARCH': bic_garch,
        'BIC_Jump': bic_jump,
        'LR': LR,
        'p_value': pval,
        'alpha_GARCH': garch['params'][2],
        'alpha_Jump': garch_jump['params'][2],
        'alpha_drop': garch['params'][2] - garch_jump['params'][2],
        'lambda': garch_jump['params'][4],
        'theta': garch_jump['params'][5],
        'delta': garch_jump['params'][6],
        'Oil_Exporter': country in oil_exporters
    })

    sig = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
    print(f"{country}: LR={LR:.1f}, p={pval:.4f}{sig}, lambda={garch_jump['params'][4]:.3f}")

cds_results_df = pd.DataFrame(cds_results)
print("\n")
print(cds_results_df.to_string())

Brazil: LR=208.0, p=0.0000***, lambda=0.039
Chile: LR=380.6, p=0.0000***, lambda=0.472
China: LR=558.8, p=0.0000***, lambda=0.284
Colombia: LR=356.9, p=0.0000***, lambda=0.235
Czechia: LR=4147.1, p=0.0000***, lambda=0.030
Egypt: LR=2123.1, p=0.0000***, lambda=0.030
Greece: LR=4629.2, p=0.0000***, lambda=0.069
Hungary: LR=4050.1, p=0.0000***, lambda=0.030
India: SKIPPED (52% zeros)
Indonesia: LR=535.7, p=0.0000***, lambda=0.352
South Korea: LR=557.6, p=0.0000***, lambda=0.076
Kuwait: SKIPPED (39% zeros)
Malaysia: LR=1203.7, p=0.0000***, lambda=1.494
Mexico: LR=271.5, p=0.0000***, lambda=0.215
Peru: LR=434.4, p=0.0000***, lambda=0.269
Philippines: LR=410.1, p=0.0000***, lambda=0.103
Poland: LR=5143.5, p=0.0000***, lambda=0.062
Qatar: LR=1191.8, p=0.0000***, lambda=0.030
Saudi Arabia: LR=1160.8, p=0.0000***, lambda=0.053
South Africa: LR=212.3, p=0.0000***, lambda=0.040
Taiwan: SKIPPED (100% zeros)
Thailand: LR=-490.9, p=1.0000, lambda=0.898
Turkey: LR=448.6, p=0.0000***, lambda=0.145
Abu

### 4.3 MSCI Country Index Series

We repeat the same GARCH vs. GARCH-Jump comparison on MSCI country equity indices. This serves as a cross-check: if jumps in equity markets also differ systematically between oil exporters and controls, it strengthens the argument that oil dynamics drive discontinuities across sovereign asset proxies.

**Note:** UAE has an MSCI index (`United Arab Emirates`) but no single CDS — CDS data is split into Abu Dhabi and Dubai.

In [66]:
msci_results = []

# For MSCI we use the original list (with "United Arab Emirates")
for country in MSCI_EM_constituent_list:
    if country not in msci_returns.columns:
        print(f"{country}: NOT FOUND in MSCI data — skipped")
        continue

    ret = msci_returns[country].dropna().values

    if len(ret) < 200:
        print(f"{country}: SKIPPED (only {len(ret)} obs)")
        continue

    zero_pct = (ret == 0).sum() / len(ret)
    if zero_pct > 0.3:
        print(f"{country}: SKIPPED ({zero_pct*100:.0f}% zeros)")
        continue

    if np.var(ret) < 1e-10:
        print(f"{country}: SKIPPED (no variance)")
        continue

    garch = estimate_garch(ret)
    garch_jump = estimate_garch_jump(ret)

    T = len(ret)
    k_garch, k_jump = 4, 7

    LR = 2 * (garch_jump['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    aic_garch = -2 * garch['loglik'] + 2 * k_garch
    bic_garch = -2 * garch['loglik'] + k_garch * np.log(T)
    aic_jump  = -2 * garch_jump['loglik'] + 2 * k_jump
    bic_jump  = -2 * garch_jump['loglik'] + k_jump * np.log(T)

    # Map UAE to oil exporter
    is_oil = country in oil_exporters or country == 'United Arab Emirates'

    msci_results.append({
        'Country': country,
        'T': T,
        'GARCH_LL': garch['loglik'],
        'Jump_LL': garch_jump['loglik'],
        'AIC_GARCH': aic_garch,
        'AIC_Jump': aic_jump,
        'BIC_GARCH': bic_garch,
        'BIC_Jump': bic_jump,
        'LR': LR,
        'p_value': pval,
        'alpha_GARCH': garch['params'][2],
        'alpha_Jump': garch_jump['params'][2],
        'alpha_drop': garch['params'][2] - garch_jump['params'][2],
        'lambda': garch_jump['params'][4],
        'theta': garch_jump['params'][5],
        'delta': garch_jump['params'][6],
        'Oil_Exporter': is_oil
    })

    sig = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
    print(f"{country}: LR={LR:.1f}, p={pval:.4f}{sig}, lambda={garch_jump['params'][4]:.3f}")

msci_results_df = pd.DataFrame(msci_results)
print("\n")
print(msci_results_df.to_string())

Brazil: LR=132.8, p=0.0000***, lambda=0.037
Chile: LR=110.4, p=0.0000***, lambda=0.037
China: LR=102.7, p=0.0000***, lambda=0.039
Colombia: LR=153.9, p=0.0000***, lambda=0.045
Czechia: LR=117.3, p=0.0000***, lambda=0.030
Egypt: LR=1940.2, p=0.0000***, lambda=0.045
Greece: LR=454.3, p=0.0000***, lambda=0.040
Hungary: LR=126.7, p=0.0000***, lambda=0.030
India: LR=216.1, p=0.0000***, lambda=0.030
Indonesia: LR=153.4, p=0.0000***, lambda=0.030
South Korea: LR=65.8, p=0.0000***, lambda=0.030
Kuwait: LR=663.7, p=0.0000***, lambda=0.030
Malaysia: LR=151.1, p=0.0000***, lambda=0.030
Mexico: LR=114.2, p=0.0000***, lambda=0.031
Peru: LR=99.2, p=0.0000***, lambda=0.030
Philippines: LR=112.2, p=0.0000***, lambda=0.030
Poland: LR=91.6, p=0.0000***, lambda=0.030
Qatar: LR=578.6, p=0.0000***, lambda=0.030
Saudi Arabia: LR=769.8, p=0.0000***, lambda=0.030
South Africa: LR=96.5, p=0.0000***, lambda=0.043
Taiwan: LR=185.2, p=0.0000***, lambda=0.030
Thailand: LR=158.8, p=0.0000***, lambda=0.030
Turkey: L

---

## 5. Extract Jump Parameters

From the estimated GARCH-Jump model, we obtain the three jump parameters that feed directly into the CCA model extension:

| Parameter | Interpretation | Example |
|-----------|---------------|---------|
| $\hat{\lambda}$ | Jump intensity (avg. jumps per period) | $\hat{\lambda} = 0.05$ → one jump every ~20 days |
| $\hat{\theta}$ | Mean jump size | Negative for oil crash dynamics |
| $\hat{\delta}^2$ | Jump size variance | Dispersion of jump magnitudes |

These map directly into the sovereign asset process of the extended CCA model:

$$dA = \mu_A \cdot A \cdot dt + \sigma_A \cdot A \cdot dW + J \cdot A \cdot dN$$

where $N$ is a Poisson process with intensity $\lambda$ and $J \sim N(\theta, \delta^2)$.

In [67]:
# Summary table: CDS jump parameters
print("=" * 80)
print("CDS — JUMP PARAMETER SUMMARY")
print("=" * 80)
cols = ['Country', 'Oil_Exporter', 'LR', 'p_value', 'lambda', 'theta', 'delta', 'alpha_GARCH', 'alpha_Jump', 'alpha_drop']
print(cds_results_df[cols].sort_values('Oil_Exporter', ascending=False).to_string(index=False))

print("\n")
print("=" * 80)
print("MSCI — JUMP PARAMETER SUMMARY")
print("=" * 80)
print(msci_results_df[cols].sort_values('Oil_Exporter', ascending=False).to_string(index=False))

CDS — JUMP PARAMETER SUMMARY
     Country  Oil_Exporter          LR  p_value   lambda     theta    delta  alpha_GARCH  alpha_Jump  alpha_drop
      Brazil          True  208.019953      0.0 0.038567  0.008029 0.074076     0.066651    0.071851   -0.005201
   Indonesia          True  535.698838      0.0 0.351722  0.007070 0.035320     0.141344    0.087512    0.053832
   Abu Dhabi          True 1698.783296      0.0 0.039989 -0.003812 0.074627     0.050000    0.050731   -0.000731
Saudi Arabia          True 1160.756313      0.0 0.053061 -0.001772 0.081883     0.130572    0.057882    0.072690
       Qatar          True 1191.839787      0.0 0.030011 -0.009992 0.030041     0.050000    0.050002   -0.000002
    Malaysia          True 1203.687209      0.0 1.493934 -0.000256 0.023975     0.067838    0.000001    0.067837
      Mexico          True  271.501944      0.0 0.215050  0.008781 0.041328     0.064974    0.097015   -0.032042
       Egypt          True 2123.110961      0.0 0.030000 -0.010000 

---

## 6. Cross-Country Comparison

We compare the jump characterization results across oil-exporter and control groups. The key question: do oil exporters show systematically stronger evidence of jumps?

| Metric | Oil Exporters vs. Controls |
|--------|---------------------------|
| Likelihood improvement | Larger $LR$ statistic for oil exporters? |
| Jump intensity $\hat{\lambda}$ | Higher for oil exporters? |
| Jump magnitude $|\hat{\theta}|$ | Larger for oil exporters? |
| $\alpha$ reduction | More dramatic drop for oil exporters? |

This cross-country comparison constitutes the first layer of the difference-in-differences argument, established at the empirical characterization stage before proceeding to the structural CCA model.

### 6.1 CDS: Exporters vs. Controls

In [68]:
print("CDS — Mean jump parameters by group:")
print(cds_results_df.groupby('Oil_Exporter')[['LR', 'lambda', 'theta', 'delta', 'alpha_drop']].mean())
print()

exporters = cds_results_df[cds_results_df['Oil_Exporter']]
controls  = cds_results_df[~cds_results_df['Oil_Exporter']]

for metric in ['LR', 'lambda', 'alpha_drop']:
    t, p = ttest_ind(exporters[metric].dropna(), controls[metric].dropna())
    u, p_mw = mannwhitneyu(exporters[metric].dropna(), controls[metric].dropna(), alternative='greater')
    print(f"{metric}:  Exporters={exporters[metric].mean():.4f}  Controls={controls[metric].mean():.4f}  "
          f"t={t:.2f} p={p:.4f}  MW-U={u:.1f} p={p_mw:.4f}")

CDS — Mean jump parameters by group:
                       LR    lambda     theta     delta  alpha_drop
Oil_Exporter                                                       
False         1706.787509  0.206483  0.004116  0.051871    0.001804
True          1226.338088  0.253502  0.000538  0.050938    0.016685

LR:  Exporters=1226.3381  Controls=1706.7875  t=-0.66 p=0.5160  MW-U=59.0 p=0.5394
lambda:  Exporters=0.2535  Controls=0.2065  t=0.31 p=0.7614  MW-U=50.0 p=0.7556
alpha_drop:  Exporters=0.0167  Controls=0.0018  t=0.89 p=0.3858  MW-U=77.0 p=0.1383


### 6.2 MSCI: Exporters vs. Controls

In [69]:
print("MSCI — Mean jump parameters by group:")
print(msci_results_df.groupby('Oil_Exporter')[['LR', 'lambda', 'theta', 'delta', 'alpha_drop']].mean())
print()

exporters_m = msci_results_df[msci_results_df['Oil_Exporter']]
controls_m  = msci_results_df[~msci_results_df['Oil_Exporter']]

for metric in ['LR', 'lambda', 'alpha_drop']:
    t, p = ttest_ind(exporters_m[metric].dropna(), controls_m[metric].dropna())
    u, p_mw = mannwhitneyu(exporters_m[metric].dropna(), controls_m[metric].dropna(), alternative='greater')
    print(f"{metric}:  Exporters={exporters_m[metric].mean():.4f}  Controls={controls_m[metric].mean():.4f}  "
          f"t={t:.2f} p={p:.4f}  MW-U={u:.1f} p={p_mw:.4f}")

MSCI — Mean jump parameters by group:
                      LR    lambda     theta     delta  alpha_drop
Oil_Exporter                                                      
False         163.560761  0.033405 -0.010611  0.035662   -0.003582
True          513.830959  0.033837 -0.011361  0.033712    0.001617

LR:  Exporters=513.8310  Controls=163.5608  t=2.30 p=0.0315  MW-U=113.0 p=0.0064
lambda:  Exporters=0.0338  Controls=0.0334  t=0.19 p=0.8511  MW-U=78.0 p=0.3303
alpha_drop:  Exporters=0.0016  Controls=-0.0036  t=0.75 p=0.4596  MW-U=83.0 p=0.2321


---

## 7. Co-Jump Analysis (Oil × CDS)

Beyond testing for jumps *within* each series, we test whether jumps in oil and sovereign CDS tend to **co-occur**. We use GARCH-standardized residuals to flag jumps, then build a contingency table for each country and test independence with Fisher's exact test.

In [70]:
@njit
def get_garch_variance(params, returns):
    """Extract conditional variance series from GARCH(1,1)."""
    mu, omega, alpha, beta = params
    T = len(returns)

    h = np.zeros(T)
    h[0] = np.var(returns)

    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]

    return h


def identify_jumps(returns, threshold=2.5):
    """
    Identify jumps using GARCH-standardized residuals.
    A jump is flagged when |z_t| > threshold.

    Returns
    -------
    pd.Series of booleans with same index as returns.
    """
    ret = returns.dropna().values
    garch = estimate_garch(ret)
    h = get_garch_variance(garch['params'], ret)
    mu = garch['params'][0]
    z = (ret - mu) / np.sqrt(h)
    jumps = np.abs(z) > threshold

    return pd.Series(jumps, index=returns.dropna().index, name=returns.name)


def cojump_test(oil_jumps, cds_jumps, country):
    """
    Test for co-jumps between oil and CDS.
    H0: Jumps are independent.
    H1: Jumps co-occur more than chance.
    """
    common_idx = oil_jumps.index.intersection(cds_jumps.index)
    oil_j = oil_jumps.loc[common_idx].values
    cds_j = cds_jumps.loc[common_idx].values

    n = len(common_idx)

    a = ((oil_j) & (cds_j)).sum()       # Both jump
    b = ((~oil_j) & (cds_j)).sum()      # Only CDS jumps
    c = ((oil_j) & (~cds_j)).sum()      # Only oil jumps
    d = ((~oil_j) & (~cds_j)).sum()     # Neither jumps

    n_oil = oil_j.sum()
    n_cds = cds_j.sum()
    expected = (n_oil / n) * (n_cds / n) * n if n > 0 else 0

    contingency = [[a, b], [c, d]]
    odds_ratio, fisher_p = fisher_exact(contingency)

    cojump_rate = a / n_oil if n_oil > 0 else np.nan

    return {
        'Country': country,
        'N': n,
        'N_oil_jumps': int(n_oil),
        'N_cds_jumps': int(n_cds),
        'N_cojumps': int(a),
        'Expected': expected,
        'Cojump_rate': cojump_rate,
        'Odds_ratio': odds_ratio,
        'Fisher_p': fisher_p
    }

In [71]:
# Identify oil jumps (Brent)
brent_returns = oil_returns['Brent']
oil_jumps = identify_jumps(brent_returns, threshold=2.5)
print(f"Oil (Brent) jumps: {oil_jumps.sum()} / {len(oil_jumps)} ({oil_jumps.mean()*100:.1f}%)")

# CDS jumps + co-jump test for each country
cds_jumps_dict = {}
cojump_results = []

for country in MSCI_EM_constituent_list_clean:
    if country not in cds_returns.columns:
        continue

    ret = cds_returns[country].dropna()

    if len(ret) < 200:
        continue
    if (ret == 0).sum() / len(ret) > 0.3:
        continue

    cds_jumps_dict[country] = identify_jumps(ret, threshold=2.5)
    res = cojump_test(oil_jumps, cds_jumps_dict[country], country)
    cojump_results.append(res)

cojump_df = pd.DataFrame(cojump_results)
cojump_df['Oil_Exporter'] = cojump_df['Country'].isin(oil_exporters)

print("\n" + "=" * 80)
print("CO-JUMP RESULTS")
print("=" * 80)
print(cojump_df.to_string(index=False))

Oil (Brent) jumps: 77 / 2868 (2.7%)

CO-JUMP RESULTS
     Country    N  N_oil_jumps  N_cds_jumps  N_cojumps  Expected  Cojump_rate  Odds_ratio  Fisher_p  Oil_Exporter
      Brazil 1968           48           50          4  1.219512     0.083333    3.703557  0.031539          True
       Chile 1968           48           52          4  1.268293     0.083333    3.545455  0.035784         False
       China 1968           48           51          2  1.243902     0.041667    1.660160  0.355288         False
    Colombia 1968           48           55          5  1.341463     0.104167    4.348837  0.009720          True
     Czechia 1968           48           52          3  1.268293     0.062500    2.545578  0.131256         False
       Egypt 1968           48           57          4  1.390244     0.083333    3.202401  0.047796          True
      Greece 1968           48           54          1  1.317073     0.020833    0.749498  1.000000         False
     Hungary 1968           48     

### 7.1 Differential Co-Jump Test

In [72]:
print("Mean co-jump rate by group:")
print(cojump_df.groupby('Oil_Exporter')[['Cojump_rate', 'Odds_ratio']].mean())

exporters_cj = cojump_df[cojump_df['Oil_Exporter']]['Cojump_rate'].dropna()
controls_cj  = cojump_df[~cojump_df['Oil_Exporter']]['Cojump_rate'].dropna()

tstat, ttest_p = ttest_ind(exporters_cj, controls_cj)
print(f"\nT-test for difference in co-jump rates:")
print(f"  Exporters mean: {exporters_cj.mean():.3f}")
print(f"  Controls mean:  {controls_cj.mean():.3f}")
print(f"  t-statistic:    {tstat:.3f}")
print(f"  p-value:        {ttest_p:.4f}")

ustat, mw_p = mannwhitneyu(exporters_cj, controls_cj, alternative='greater')
print(f"\nMann-Whitney U test (exporters > controls):")
print(f"  U-statistic: {ustat:.1f}")
print(f"  p-value:     {mw_p:.4f}")

Mean co-jump rate by group:
              Cojump_rate  Odds_ratio
Oil_Exporter                         
False             0.06250    2.531843
True              0.09375    3.680415

T-test for difference in co-jump rates:
  Exporters mean: 0.094
  Controls mean:  0.062
  t-statistic:    2.593
  p-value:        0.0174

Mann-Whitney U test (exporters > controls):
  U-statistic: 95.5
  p-value:     0.0088


## Weekly jump in MSCI

In [76]:
# MSCI country indices (daily → weekly)
MSCI_data = pd.read_csv('data/processed/MSCI_indices/mscicountryindex.csv')  # adjust path as needed
MSCI_data['Date'] = pd.to_datetime(MSCI_data['Date'])
MSCI_data.set_index('Date', inplace=True)
MSCI_data = MSCI_data.resample('W-FRI').last()
msci_returns = np.log(MSCI_data / MSCI_data.shift(1)).dropna()

In [77]:
msci_results = []

# For MSCI we use the original list (with "United Arab Emirates")
for country in MSCI_EM_constituent_list:
    if country not in msci_returns.columns:
        print(f"{country}: NOT FOUND in MSCI data — skipped")
        continue

    ret = msci_returns[country].dropna().values

    if len(ret) < 200:
        print(f"{country}: SKIPPED (only {len(ret)} obs)")
        continue

    zero_pct = (ret == 0).sum() / len(ret)
    if zero_pct > 0.3:
        print(f"{country}: SKIPPED ({zero_pct*100:.0f}% zeros)")
        continue

    if np.var(ret) < 1e-10:
        print(f"{country}: SKIPPED (no variance)")
        continue

    garch = estimate_garch(ret)
    garch_jump = estimate_garch_jump(ret)

    T = len(ret)
    k_garch, k_jump = 4, 7

    LR = 2 * (garch_jump['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    aic_garch = -2 * garch['loglik'] + 2 * k_garch
    bic_garch = -2 * garch['loglik'] + k_garch * np.log(T)
    aic_jump  = -2 * garch_jump['loglik'] + 2 * k_jump
    bic_jump  = -2 * garch_jump['loglik'] + k_jump * np.log(T)

    # Map UAE to oil exporter
    is_oil = country in oil_exporters or country == 'United Arab Emirates'

    msci_results.append({
        'Country': country,
        'T': T,
        'GARCH_LL': garch['loglik'],
        'Jump_LL': garch_jump['loglik'],
        'AIC_GARCH': aic_garch,
        'AIC_Jump': aic_jump,
        'BIC_GARCH': bic_garch,
        'BIC_Jump': bic_jump,
        'LR': LR,
        'p_value': pval,
        'alpha_GARCH': garch['params'][2],
        'alpha_Jump': garch_jump['params'][2],
        'alpha_drop': garch['params'][2] - garch_jump['params'][2],
        'lambda': garch_jump['params'][4],
        'theta': garch_jump['params'][5],
        'delta': garch_jump['params'][6],
        'Oil_Exporter': is_oil
    })

    sig = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
    print(f"{country}: LR={LR:.1f}, p={pval:.4f}{sig}, lambda={garch_jump['params'][4]:.3f}")

msci_results_df = pd.DataFrame(msci_results)
print("\n")
print(msci_results_df.to_string())

Brazil: LR=36.9, p=0.0000***, lambda=0.105
Chile: LR=18.4, p=0.0004***, lambda=0.057
China: LR=21.2, p=0.0001***, lambda=0.041
Colombia: LR=43.7, p=0.0000***, lambda=0.049
Czechia: LR=51.5, p=0.0000***, lambda=0.040
Egypt: LR=328.4, p=0.0000***, lambda=0.144
Greece: LR=50.4, p=0.0000***, lambda=0.051
Hungary: LR=39.0, p=0.0000***, lambda=0.042
India: LR=30.8, p=0.0000***, lambda=0.233
Indonesia: LR=91.7, p=0.0000***, lambda=0.167
South Korea: LR=14.0, p=0.0029***, lambda=0.038
Kuwait: LR=130.6, p=0.0000***, lambda=0.061
Malaysia: LR=60.4, p=0.0000***, lambda=0.041
Mexico: LR=50.7, p=0.0000***, lambda=0.014
Peru: LR=30.6, p=0.0000***, lambda=0.040
Philippines: LR=26.1, p=0.0000***, lambda=0.049
Poland: LR=49.0, p=0.0000***, lambda=0.019
Qatar: LR=44.4, p=0.0000***, lambda=0.043
Saudi Arabia: LR=131.6, p=0.0000***, lambda=0.040
South Africa: LR=31.4, p=0.0000***, lambda=0.006
Taiwan: LR=45.0, p=0.0000***, lambda=0.140
Thailand: LR=32.5, p=0.0000***, lambda=0.076
Turkey: LR=70.5, p=0.0000

In [80]:
msci_results_df[['Country','lambda','theta','delta']].to_csv('msci_jump_parameters.csv', index=False)